# Within-BRCA Confounding

Reproduces Table `tab:app-within-brca` and the Spearman correlation between peak S2 and probe AUROC (Appendix, Within-BRCA). DECAT is applied within BRCA patients only, so cancer type is constant by design. Candidate confounders from Dawood et al. (2026) are never provided to DECAT and are used only for post-hoc stratification.

**Design:** B is a random split of the evaluation pool; C draws a fraction $\alpha$ from the extreme pool E (top 25% of the C pool by FM probe score) and the remainder from the rest of the pool. 50 splits per FM × label × $\alpha$.

**Inputs:**
- `brca_{label}_fm_site_confounding_records.json` from `experiments/tcga_validation/run_within_ct_decat.py`
- `within_ct_collapse_summary.csv` from `experiments/tcga_validation/run_within_ct_collapse.py`

In [ ]:
import json
from pathlib import Path

import pandas as pd
from scipy.stats import spearmanr

from decimal import Decimal, ROUND_HALF_UP


def fmt(v: float, nd: int = 3) -> str:
    """Format with round-half-up, matching the paper (exact ties such as 0.3045 occur)."""
    return str(Decimal(f'{v:.10f}').quantize(Decimal(1).scaleb(-nd), rounding=ROUND_HALF_UP))

DATA_DIR = Path('../../data')
OUT_DIR = DATA_DIR / 'tcga'

FM_LABELS = {'titan': 'TITAN', 'conch': 'CONCH', 'uni': 'UNI',
             'hoptimus0': 'H-Optimus-0', 'openmidnight': 'OpenMidnight'}
# Label -> stratification key of its candidate confounder in the collapse summary
CONFOUNDER_KEY = {'TP53': 'ER_STATUS', 'CDH1': 'PRIMARY_DIAG', 'ER': 'CDH1_MUT', 'PR': 'ER_STATUS'}
DEFAULT_ALPHA = 0.6  # reported for FMs with 0% peak S2

records = []
for label in CONFOUNDER_KEY:
    recs = json.load(open(OUT_DIR / f'brca_{label.lower()}_fm_site_confounding_records.json'))
    for r in recs:
        r['label'] = label
    records.extend(recs)
    print(f'BRCA / {label}: {len(recs)} records')
df = pd.DataFrame(records)

collapse = pd.read_csv(OUT_DIR / 'within_ct_collapse_summary.csv')
collapse = collapse[(collapse['cancer'] == 'BRCA')
                    & (collapse['stratum_key'] == collapse['label'].map(CONFOUNDER_KEY))]

In [ ]:
rows = []
for label in CONFOUNDER_KEY:
    for fm, fm_name in FM_LABELS.items():
        d = df[(df['label'] == label) & (df['fm'] == fm)]
        s2_by_alpha = d.groupby('alpha')['verdict'].apply(lambda v: (v == 'S2').mean())
        peak = s2_by_alpha.idxmax() if s2_by_alpha.max() > 0 else None
        at = d[d['alpha'] == (DEFAULT_ALPHA if peak is None else peak)]
        c = collapse[(collapse['label'] == label) & (collapse['fm'] == fm)]
        worst = c[c['stratum'] != 'pan'].sort_values('delta').iloc[-1]
        rows.append({
            'Label': label,
            'Confounder (eta2)': f"{worst['confounder']} ({worst['eta2']:.3f})",
            'FM': fm_name,
            'Peak alpha': '---' if peak is None else f'{peak:.1f}',
            'S2': round(100 * s2_by_alpha.max()),
            'D_quantile': round(100 * at['dq_fires'].mean()),
            'P_transfer': round(100 * at['ptf_fires'].mean()),
            'Indet.': round(100 * (at['verdict'] == 'Indeterminate').mean()),
            'Transferable': round(100 * at['verdict'].str.startswith('Trans').mean()),
            'AUROC': c.loc[c['stratum'] == 'pan', 'auroc'].iloc[0],
            'Worst stratum': worst['auroc'],
            'Stratum': worst['stratum'],
            'Delta': worst['delta'],
            'Delta n_splits': worst['n_splits'],
            'Mean CV AUROC on A\'': d['auroc'].mean(),
        })

table = pd.DataFrame(rows).set_index(['Label', 'Confounder (eta2)', 'FM'])
print(table.drop(columns=['Transferable', "Mean CV AUROC on A'"])
           .to_string(float_format=fmt))
print(f"\nMax transferable at the reported alpha: {table['Transferable'].max()}%")

In [ ]:
# Peak S2 vs probe AUROC across the 20 FM x label rows
rho = spearmanr(table['S2'], table["Mean CV AUROC on A'"])
print(f'Spearman rho = {rho.statistic:.2f} (p = {rho.pvalue:.3g})')